In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup. Nothing here needs editing: the deposit root is found by
# walking up from wherever this notebook happens to be opened.
#
# NOTE: this notebook drives the spatial light modulator and the
# camera. It needs the hardware and the vendor SDKs (Holoeye SLM
# Display SDK, Canon EDSDK), which cannot be redistributed. The
# simulation cells below run without hardware; the bench cell does
# not. To reproduce the paper's figures from the deposited data,
# use gd_citl_analysis.ipynb instead.
# ─────────────────────────────────────────────────────────────
import os, sys, json
from pathlib import Path


def _deposit_root(start=None):
    """First ancestor containing code/analysis/paths.py."""
    p = (start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / "code" / "analysis" / "paths.py").exists():
            return cand
    raise FileNotFoundError(f"could not locate the deposit root from {p}")


ROOT = _deposit_root()
for _sub in ("analysis", "method"):
    _d = str(ROOT / "code" / _sub)
    if _d not in sys.path:
        sys.path.insert(0, _d)

import paths

CONFIG_DIR = paths.CONFIGS

# Holoeye SLM Display SDK, located via the environment variable its installer
# sets. Only needed by the bench cell.
_heds = os.getenv("HEDS_3_2_PYTHON_MODULES", "")
if _heds:
    sys.path.append(_heds)

_ip = get_ipython()
if "autoreload" not in _ip.extension_manager.loaded:
    _ip.run_line_magic("load_ext", "autoreload")
_ip.run_line_magic("autoreload", "2")

paths.report()



---

# SIMULATION

---

## 1. Gradient-descent simulation sweep

Produces the `simulation` source in `scored.json` and the converged holograms used as the simulation-seeded initialization. Runs without hardware.


In [ ]:
from datetime import datetime
import numpy as np
import os
import shutil
import time
import json
import torch

from config_handler import ConfigHandler
from experiment_manager import ExperimentManager
from evaluate_methods import evaluate_single_image_multilevel

IMAGE_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")


def list_image_files(directory):
    """Every image in `directory`, sorted, so run order is deterministic."""
    d = Path(directory)
    return [str(p) for p in sorted(d.iterdir())
            if p.suffix.lower() in IMAGE_SUFFIXES]


config_file = "simulation_comparison"
config = ConfigHandler.load(config_file, search_paths=[str(CONFIG_DIR)])

multiple_images_path = paths.resolve(config.paths.multiple_images)
save_path = paths.resolve(config.paths.save)
save_path.mkdir(parents=True, exist_ok=True)

image_files = list_image_files(multiple_images_path)
print(f"Found {len(image_files)} image files in {multiple_images_path}")

# -----------------------------
# Torch device + determinism
# -----------------------------
seed = 0
torch.manual_seed(seed)
np.random.seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

timestamp = datetime.now()

all_results = []

for i, image_file in enumerate(image_files):
    image_stem = os.path.splitext(os.path.basename(image_file))[0]

    print(f"\n{'='*80}")
    print(f"Starting image {i+1}/{len(image_files)}: {image_stem}")
    print(f"{'='*80}\n")

    experiment = ExperimentManager(
        name=f"{image_stem}_{config.experiment.name}",
        base_dir=str(save_path),
        overwrite=config.experiment.overwrite,
    )

    out_dir = os.path.join(experiment.dir, "outputs")
    os.makedirs(out_dir, exist_ok=True)

    experiment.log(f"Experiment Details:\r\nExperiment: {config.experiment.description}")
    experiment.log(f"Date: {timestamp}")

    # snapshot the config alongside the run, so a result always carries the
    # settings that produced it
    shutil.copy2(os.path.join(str(CONFIG_DIR), config_file + ".json"),
                 os.path.join(experiment.dir, "run_configuration.json"))

    # -----------------------------
    # Forward-model settings
    # -----------------------------
    H, W = config.slm_shape
    target_formulations = tuple(getattr(config.experiment, "target_formulations", ["amplitude"]))
    pad_factor = config.physics.pad_factor
    apply_sinc = bool(config.physics.apply_sinc)
    fill_factor = config.physics.fill_factor
    # fill_is_areal defaults to True (areal, matching the modulator datasheet's
    # fill-factor convention) via getattr, so older config snapshots that predate
    # this key still load.
    fill_is_areal = bool(getattr(config.physics, "fill_is_areal", True))

    experiment.log(
        f"Physics settings: pad_factor={pad_factor}, apply_sinc={apply_sinc}, "
        f"fill_factor={fill_factor}, fill_is_areal={fill_is_areal}"
    )

    t0 = time.time()

    summary = evaluate_single_image_multilevel(
        image_path=image_file,
        out_dir=out_dir,
        device=device,
        gd_iterations_list=tuple(config.gd.iterations_list),
        gd_kwargs={
            "init_phase": config.gd.init_phase,
            "lr": config.gd.lr,
            "optimizer_name": config.gd.optimizer_name,
            "loss_type": config.gd.loss_type,
            "tv_weight": config.gd.tv_weight,
            "lowpass": config.gd.lowpass,
            "lpf_sigma": config.gd.lpf_sigma,
            "add_carrier": config.gd.add_carrier,
            "show": config.gd.show,
        },
        target_formulations=target_formulations,
        # These four are what make the forward-model settings above do anything.
        # Omitting them silently runs the IDEAL model regardless of what
        # physics.* says in the config.
        pad_factor=pad_factor,
        apply_sinc=apply_sinc,
        fill_factor=fill_factor,
        fill_is_areal=fill_is_areal,
    )

    experiment.log(f"target_formulations used for this run: {target_formulations}")

    elapsed = time.time() - t0
    experiment.log(f"Finished {image_stem} in {elapsed:.3f} s")

    image_result = {
        "image_name": image_stem,
        "image_path": image_file,
        "elapsed_seconds": elapsed,
        "pad_factor": pad_factor,
        "apply_sinc": apply_sinc,
        "fill_factor": fill_factor,
        "fill_is_areal": fill_is_areal,
        "summary": summary,
    }
    all_results.append(image_result)

    with open(os.path.join(out_dir, "run_summary.json"), "w") as f:
        json.dump(image_result, f, indent=2)

# -----------------------------
# Aggregate. gdcitl.simulation_sweep() reads this file.
# -----------------------------
aggregate_path = os.path.join(str(save_path), f"{config.experiment.name}_all_results.json")
with open(aggregate_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\nSaved aggregate results to: {aggregate_path}")


## 2. GD+CITL Aberration Stress Test (Simulation)
Inject a known Zernike aberration into the *measurement* while GD's own forward model stays clean - GD only senses the aberration through camera-in-the-loop feedback, mirroring a real bench. Question: can CITL recover the target, and up to what aberration magnitude? Two sweeps (per-mode diagnostic, combined headline) across 0 to 1.0λ RMS, on the faithful forward model (pad=2, sinc on).

In [ ]:
"""
GD+CITL far-field aberration stress test (PURE SIMULATION).

Emulates the GD+CITL experiment pipeline (camera-in-the-loop gradient descent),
but replaces the real Canon/SLM hardware with a SYNTHETIC ABERRATED CAMERA.

The one and only conceptual difference from the hardware experiment is the
camera_source passed to gd.run():
  * experiment: gd_camera_source -> capture_for_phase(...) -> real Canon photo
  * this test:  gd_camera_source -> make_aberrated_camera_source(...) ->
                | faithful_forward( exp(i*(phase + zernike_aberration)) ) |^2
Everything else about how gd.run() is invoked (same optimizer, lr, iterations,
target_formulation, faithful physics settings) is identical, so any difference
in the result is attributable to the injected aberration alone.

The optimizer's OWN forward model (its gradients) stays un-aberrated - it is
blind to the aberration and senses it only through the synthetic measurement,
exactly as GD would only sense a real bench's aberration through a real camera.
This is the whole premise: can camera-in-the-loop feedback recover the target
despite the optimizer's model being wrong (missing the aberration)?

Pure simulation => NO zero-order/DC spike (a physical bench artifact), so
gd.run is called with dc_mask=None.

Two sweeps (see config):
  1. per-mode  : each of 9 Noll modes swept independently across magnitudes.
  2. combined  : 6 image-degrading modes at equal RMS, total-RMS swept.

Intended to be run from a Jupyter notebook or as a script. Fill in the FILL_IN
paths in the config. Depends on: physics.py, gd.py, stats_torch.py,
evaluate_methods.py (for scoring), zernike_aberrations.py, and your project's
utils/config_handler/experiment_manager.
"""
from datetime import datetime
import os
import json
import time

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from config_handler import ConfigHandler
from experiment_manager import ExperimentManager
from stats_torch import normalize_intensity_sum
from gd import GradientDescentHologram
from evaluate_methods import reconstruct_from_phase_np, compute_metrics, _target_eval_for_padding

from zernike_aberrations import (
    aberration_phase_map,
    make_aberrated_camera_source,
    NOLL_NAMES,
)


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
config_file = "gd_zernike_stress_test"
config = ConfigHandler.load(config_file, search_paths=[str(CONFIG_DIR)])

save_path = paths.resolve(config.paths.save)
save_path.mkdir(parents=True, exist_ok=True)
images_path = paths.resolve(config.paths.images)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Physics (faithful model) - passed to BOTH gd.run and the synthetic camera.
PAD_FACTOR = config.physics.pad_factor
APPLY_SINC = bool(config.physics.apply_sinc)
FILL_FACTOR = config.physics.fill_factor
FILL_IS_AREAL = bool(config.physics.fill_is_areal)

# Run settings
H = config.run.height
W = config.run.width
GD_ITERATIONS = config.run.gd_iterations
GD_LR = config.run.gd_lr
GD_OPTIMIZER = config.run.gd_optimizer_name
GD_TV_WEIGHT = config.run.gd_tv_weight
GD_INIT_PHASE = config.run.gd_init_phase
CITL_INIT_PHASE = config.run.citl_init_phase
TARGET_FORMULATION = config.run.target_formulation
SEED = config.run.seed

MAGNITUDES = list(config.aberrations.magnitudes_waves_rms)
PER_MODE_ENABLED = config.aberrations.per_mode.enabled
PER_MODE_NOLL = list(config.aberrations.per_mode.noll_indices)
COMBINED_ENABLED = config.aberrations.combined.enabled
COMBINED_NOLL = list(config.aberrations.combined.noll_indices)

# Optional near-publication-quality figure saving: TARGET | aberrated-system
# view of GD's final hologram | GD+CITL's clean-model reconstruction, per
# condition, for a SELECTED subset of images/magnitudes (saving every one of
# the ~1600 conditions would be excessive; this is meant for the handful of
# conditions that will actually go in the paper).
SAVE_VISUALS = bool(getattr(config, "save_visuals", None) and config.save_visuals.enabled)
if SAVE_VISUALS:
    VISUAL_IMAGE_STEMS = set(config.save_visuals.image_stems)
    VISUAL_MAGNITUDES = set(config.save_visuals.magnitudes_waves_rms)
    VISUAL_MODE_NOLL = set(getattr(config.save_visuals, "noll_indices", PER_MODE_NOLL))
    VISUAL_SAVE_COMBINED = bool(getattr(config.save_visuals, "save_combined", True))
    VISUAL_DPI = int(getattr(config.save_visuals, "dpi", 200))
else:
    VISUAL_IMAGE_STEMS = set()
    VISUAL_MAGNITUDES = set()
    VISUAL_MODE_NOLL = set()
    VISUAL_SAVE_COMBINED = False
    VISUAL_DPI = 200


# ---------------------------------------------------------------------------
# Target preparation (identical convention to the GD+CITL experiment)
# ---------------------------------------------------------------------------
def prepare_target(image_path):
    """Mirror the experiment's target prep exactly."""
    img = Image.open(image_path).convert("L")
    X = np.asarray(img, dtype=np.float32)
    if X.shape != (H, W):
        raise ValueError(f"Target image shape {X.shape} != expected ({H},{W})")

    X_clipped = np.clip(X, 0.0, None)
    if TARGET_FORMULATION == "amplitude":
        target_optim_np = np.sqrt(X_clipped).astype(np.float32)
    else:
        target_optim_np = X_clipped.astype(np.float32)

    # target_eval_np: normalized intensity for scoring (native resolution).
    X_batched = X.reshape(1, H, W)
    target_eval_t = torch.from_numpy(X_batched).to(device=device, dtype=torch.float32)
    target_eval_t = normalize_intensity_sum(target_eval_t)
    target_eval_np = target_eval_t[0].detach().cpu().numpy().astype(np.float32)
    return target_optim_np, target_eval_np


def _propagate_and_score(phase_np, camera_source, iteration_index, target_eval_scaled, device):
    """
    Propagate a given phase through a camera_source (the TRUE, aberrated
    system) and score the result against the target. Used for BOTH CITL's
    final hologram and the naive (no-CITL) baseline's hologram - same single
    computation path, just called on different phases, so there is still only
    ONE place that computes "what does the aberrated system show for this
    phase" (see run_one_condition's docstring on why re-using camera_source
    rather than reimplementing this matters).

    Returns: (view_np, metrics_dict) where view_np is sum=1 normalized
    (matching target_eval_scaled's convention) and metrics_dict is
    compute_metrics's output (psnr/ssim/nmse/mse).
    """
    phase_t = torch.from_numpy(np.asarray(phase_np, dtype=np.float32)).to(device)
    with torch.no_grad():
        I_ab = camera_source(phase_t, iteration_index)
    view_np = I_ab[0].detach().cpu().numpy().astype(np.float32)
    view_np = view_np / (view_np.sum() + 1e-12)  # unit conversion to sum=1, see run_one_condition
    metrics = compute_metrics(view_np, target_eval_scaled)
    return view_np, metrics


def run_naive_baseline(target_optim_np):
    """
    Run plain GD ONCE per image, with NO camera feedback (camera_source=None)
    - i.e. exactly what the bench would produce with no CITL correction at
    all. This is aberration-INDEPENDENT (the optimizer never sees any
    aberration either way when camera_source=None), so it is computed once per
    image and re-used (re-scored against each condition's own aberration) by
    every call to run_one_condition, rather than re-running GD once per
    condition - that would be 1600x more compute for a result that cannot
    change per condition.

    This is the "what if we didn't use CITL" baseline: the SAME naive
    hologram, scored against each condition's TRUE aberrated system, tells you
    how much CITL is actually buying you over doing nothing, at every
    magnitude - the number the paper's core claim rests on, not just "CITL
    recovers to PSNR=X" in isolation.

    Returns: dict with phase_radians and its own clean-model metrics (a single
    aberration-independent reference point: how good is plain GD when there is
    no aberration at all).
    """
    gd_out = GradientDescentHologram().run(
        target_optim_np,
        iterations=GD_ITERATIONS,
        init_phase=GD_INIT_PHASE,
        lr=GD_LR,
        optimizer_name=GD_OPTIMIZER,
        tv_weight=GD_TV_WEIGHT,
        device=device,
        target_formulation=TARGET_FORMULATION,
        camera_source=None,  # NO CITL - the naive/no-correction baseline
        dc_mask=None,
        seed=SEED,
        pad_factor=PAD_FACTOR,
        apply_sinc=APPLY_SINC,
        fill_factor=FILL_FACTOR,
        fill_is_areal=FILL_IS_AREAL,
    )
    return {"phase_radians": gd_out["phase_radians"]}


# ---------------------------------------------------------------------------
# One GD+CITL run under a given aberration, scored consistently
# ---------------------------------------------------------------------------
def run_one_condition(target_optim_np, target_eval_np, aberration_phase_radians,
                      naive_phase_np=None):
    """
    Run GD+CITL once with the synthetic aberrated camera, return metrics.

    Every condition now reports THREE scores against the target, answering
    three different questions:
      1. psnr/ssim/nmse/mse - CITL's hologram, scored on the
         CLEAN model. "What does the optimizer THINK it produced." Matches how
         every other method in the project is scored (reconstruct_from_phase_np).
      2. aberrated_psnr/... (etc) - CITL's hologram, scored on the TRUE
         aberrated system (i.e. what the aberrated_view_np panel actually
         shows). "What would the real bench actually show, using CITL."
      3. naive_aberrated_psnr/... (etc) - the NAIVE, no-CITL hologram (same
         one for every condition on this image - see run_naive_baseline),
         scored on THIS condition's TRUE aberrated system. "What would the
         real bench show with NO correction." This is the baseline that makes
         "CITL recovers N dB over no correction, up to magnitude X" a
         well-defined, reportable claim - without it, (1) and (2) alone don't
         establish that CITL is helping, only that CITL's own result has some
         score.
    aberrated_view_np (for figures) is always computed now (previously gated
    behind compute_aberrated_view - removed, since we need its SCORE on every
    condition regardless of whether we're also saving a figure for it; the
    extra forward pass is cheap relative to the 750-iteration GD run).
    """
    # Synthetic aberrated camera: same faithful physics as gd.run's clean model,
    # differing ONLY by the added aberration.
    camera_source = make_aberrated_camera_source(
        aberration_phase_radians=aberration_phase_radians,
        pad_factor=PAD_FACTOR, apply_sinc=APPLY_SINC,
        fill_factor=FILL_FACTOR, fill_is_areal=FILL_IS_AREAL,
        device=device,
    )

    citl_seed = (naive_phase_np if (CITL_INIT_PHASE == "warm_start" and naive_phase_np is not None)
                else CITL_INIT_PHASE)

    gd_out = GradientDescentHologram().run(
        target_optim_np,
        iterations=GD_ITERATIONS,
        init_phase=citl_seed,
        lr=GD_LR,
        optimizer_name=GD_OPTIMIZER,
        tv_weight=GD_TV_WEIGHT,
        device=device,
        target_formulation=TARGET_FORMULATION,
        camera_source=camera_source,
        dc_mask=None,  # pure sim: no physical DC spike
        seed=SEED,
        pad_factor=PAD_FACTOR,
        apply_sinc=APPLY_SINC,
        fill_factor=FILL_FACTOR,
        fill_is_areal=FILL_IS_AREAL,
    )

    # Score 1: CITL's hologram on the CLEAN model (same path as all other methods).
    recon_np = reconstruct_from_phase_np(
        gd_out["phase_radians"], device=device,
        pad_factor=PAD_FACTOR, apply_sinc=APPLY_SINC,
        fill_factor=FILL_FACTOR, fill_is_areal=FILL_IS_AREAL,
    )
    target_eval_scaled = _target_eval_for_padding(target_eval_np, PAD_FACTOR)
    metrics = compute_metrics(recon_np, target_eval_scaled)

    # Score 2: CITL's hologram on the TRUE aberrated system (re-uses
    # camera_source - the single source of truth for "what does the aberrated
    # system produce"; see _propagate_and_score / module docstring on why we
    # don't reimplement this separately).
    aberrated_view_np, aberrated_metrics = _propagate_and_score(
        gd_out["phase_radians"], camera_source, GD_ITERATIONS, target_eval_scaled, device
    )

    # Score 3: the NAIVE (no-CITL) hologram, on THIS condition's TRUE aberrated
    # system - the "what if we did nothing" baseline.
    naive_aberrated_view_np = None
    naive_aberrated_metrics = None
    if naive_phase_np is not None:
        naive_aberrated_view_np, naive_aberrated_metrics = _propagate_and_score(
            naive_phase_np, camera_source, GD_ITERATIONS, target_eval_scaled, device
        )

    return {
        "psnr": metrics["psnr"], "ssim": metrics["ssim"],
        "nmse": metrics["nmse"], "mse": metrics["mse"],
        "aberrated_psnr": aberrated_metrics["psnr"], "aberrated_ssim": aberrated_metrics["ssim"],
        "aberrated_nmse": aberrated_metrics["nmse"], "aberrated_mse": aberrated_metrics["mse"],
        "naive_aberrated_psnr": None if naive_aberrated_metrics is None else naive_aberrated_metrics["psnr"],
        "naive_aberrated_ssim": None if naive_aberrated_metrics is None else naive_aberrated_metrics["ssim"],
        "naive_aberrated_nmse": None if naive_aberrated_metrics is None else naive_aberrated_metrics["nmse"],
        "naive_aberrated_mse": None if naive_aberrated_metrics is None else naive_aberrated_metrics["mse"],
        "best_iteration": gd_out.get("best_iteration"),
        "final_phase": gd_out["phase_radians"],  # caller decides whether to save
        "recon_np": recon_np,                    # clean-model reconstruction
        "aberrated_view_np": aberrated_view_np,   # CITL's hologram on the TRUE system
        "naive_aberrated_view_np": naive_aberrated_view_np,  # naive hologram on the TRUE system
    }


# ---------------------------------------------------------------------------
# Figure saving (opt-in, selected conditions only)
# ---------------------------------------------------------------------------
def save_condition_figure(target_eval_np, naive_aberrated_view_np, aberrated_view_np,
                          mag, aberrated_psnr, aberrated_nmse,
                          naive_psnr, naive_nmse, out_path, panel_title,
                          recon_gamma: float = 0.5, percentile: float = 99.0):
    """
    3-panel figure:
      1. Target
      2. Naive (no CITL) hologram, viewed on the TRUE aberrated system --
         shows what the aberration alone does, with no correction.
      3. GD+CITL hologram, viewed on the TRUE aberrated system - shows how
         much CITL recovers, having only ever seen the aberrated camera
         feedback (simulated here), never the aberration itself.
    Comparing panel 2's numbers to panel 3's numbers IS the claim: how much
    does CITL improve over doing nothing, at this aberration and magnitude.

    Grayscale, no axis ticks, matching the project's existing figure style
    (Fig. 1-5 in the manuscript). Saves PNG at VISUAL_DPI. Also saves the
    three raw arrays as .npy alongside, so the figure can be rebuilt/restyled
    later without re-running GD.

    ASYMMETRIC gamma (physically motivated, not stylistic): target_eval_np
    never touches hologram_intensity_from_phase, so pad_factor/apply_sinc have
    zero effect on it - displayed ALWAYS linear (gamma=1, hardcoded). The
    other two panels DO pass through the full faithful forward model, where
    apply_sinc=True concentrates energy into fewer, brighter points (verified:
    heavy-tailed even at zero aberration) - these get `recon_gamma` (default
    0.5). Set recon_gamma=1.0 for a fully literal comparison.

    Safety net: if a panel's own max is drastically below the shared
    reference, it falls back to its own percentile (labeled
    "[independently rescaled]") rather than rendering fully black.
    """
    ref_vmax = np.percentile(np.clip(target_eval_np, 0, None), percentile)
    if ref_vmax <= 1e-12:
        ref_vmax = float(np.clip(target_eval_np, 0, None).max()) + 1e-12

    def _display(img, label, gamma):
        img = np.clip(img, 0, None)
        own_vmax = np.percentile(img, percentile)
        rescaled_note = ""
        vmax = ref_vmax
        if own_vmax < 0.05 * ref_vmax and own_vmax > 1e-12:
            vmax = own_vmax
            rescaled_note = " [independently rescaled]"
        return np.clip(img / vmax, 0.0, 1.0) ** gamma, label + rescaled_note

    target_disp, target_label = _display(target_eval_np, "Target", gamma=1.0)
    naive_disp, naive_label = _display(
        naive_aberrated_view_np,
        f"Naive (no CITL)\n({panel_title}, {mag:.2f}\u03bb RMS)\nPSNR={naive_psnr:.2f} dB, NMSE={naive_nmse:.4f}",
        gamma=recon_gamma,
    )
    aberr_disp, aberr_label = _display(
        aberrated_view_np,
        f"GD+CITL\n({panel_title}, {mag:.2f}\u03bb RMS)\nPSNR={aberrated_psnr:.2f} dB, NMSE={aberrated_nmse:.4f}",
        gamma=recon_gamma,
    )

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.6), constrained_layout=True)
    for ax, img, title in zip(
        axes, [target_disp, naive_disp, aberr_disp],
        [target_label, naive_label, aberr_label],
    ):
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    fig.savefig(out_path, dpi=VISUAL_DPI, bbox_inches="tight")
    plt.close(fig)

    # Raw arrays alongside, for later re-plotting without re-running GD.
    base = os.path.splitext(out_path)[0]
    np.save(base + "_target.npy", target_eval_np)
    np.save(base + "_naive_aberrated_view.npy", naive_aberrated_view_np)
    np.save(base + "_aberrated_view.npy", aberrated_view_np)


# ---------------------------------------------------------------------------
# Image discovery
# ---------------------------------------------------------------------------
def list_images():
    exts = (".png", ".jpg", ".jpeg")
    files = sorted(
        os.path.join(images_path, f) for f in os.listdir(images_path)
        if f.lower().endswith(exts)
    )
    if config.images_subset.use_subset:
        stems = set(config.images_subset.stems)
        files = [f for f in files
                 if os.path.splitext(os.path.basename(f))[0] in stems]
    return files


# ---------------------------------------------------------------------------
# Main sweep
# ---------------------------------------------------------------------------
def main():
    image_files = list_images()
    print(f"Found {len(image_files)} images to process.")

    experiment = ExperimentManager(
        name=config.experiment.name,
        base_dir=save_path,
        overwrite=config.experiment.overwrite,
    )
    experiment.log(f"GD+CITL Zernike stress test - {datetime.now()}")
    experiment.log(f"physics: pad_factor={PAD_FACTOR}, apply_sinc={APPLY_SINC}, "
                   f"fill_factor={FILL_FACTOR}, fill_is_areal={FILL_IS_AREAL}")
    experiment.log(f"GD: iters={GD_ITERATIONS}, lr={GD_LR}, opt={GD_OPTIMIZER}, "
                   f"target_formulation={TARGET_FORMULATION}, seed={SEED}")
    experiment.log(f"magnitudes (waves RMS): {MAGNITUDES}")

    all_results = []

    for img_idx, image_file in enumerate(image_files):
        stem = os.path.splitext(os.path.basename(image_file))[0]
        print(f"\n{'='*70}\nImage {img_idx+1}/{len(image_files)}: {stem}\n{'='*70}")
        target_optim_np, target_eval_np = prepare_target(image_file)

        # ---- Naive (no-CITL) baseline: ONE run per image, aberration-independent ----
        # See run_naive_baseline docstring: this is the "what if we did nothing"
        # hologram, re-scored against every condition's own true aberrated
        # system below (not re-run per condition).
        t0 = time.perf_counter()
        naive_out = run_naive_baseline(target_optim_np)
        naive_phase_np = naive_out["phase_radians"]
        dt_naive = time.perf_counter() - t0
        naive_clean_recon = reconstruct_from_phase_np(
            naive_phase_np, device=device,
            pad_factor=PAD_FACTOR, apply_sinc=APPLY_SINC,
            fill_factor=FILL_FACTOR, fill_is_areal=FILL_IS_AREAL,
        )
        target_eval_scaled_ref = _target_eval_for_padding(target_eval_np, PAD_FACTOR)
        naive_clean_metrics = compute_metrics(naive_clean_recon, target_eval_scaled_ref)
        print(f"  [naive baseline] no-CITL, no-aberration reference: "
              f"PSNR={naive_clean_metrics['psnr']:.2f} NMSE={naive_clean_metrics['nmse']:.4f} "
              f"({dt_naive:.1f}s, computed once for this image)")

        # ---- Per-mode sweep ----
        if PER_MODE_ENABLED:
            for noll_j in PER_MODE_NOLL:
                mode_name = NOLL_NAMES.get(noll_j, f"noll{noll_j}")
                for mag in MAGNITUDES:
                    want_visual = (
                        SAVE_VISUALS and stem in VISUAL_IMAGE_STEMS
                        and mag in VISUAL_MAGNITUDES and noll_j in VISUAL_MODE_NOLL
                    )
                    aberr = aberration_phase_map(
                        grid_size=H, magnitude_waves_rms=mag,
                        noll_indices=(noll_j,),
                    )
                    t0 = time.perf_counter()
                    res = run_one_condition(
                        target_optim_np, target_eval_np, aberr,
                        naive_phase_np=naive_phase_np,
                    )
                    dt = time.perf_counter() - t0
                    rec = {
                        "image": stem, "sweep": "per_mode",
                        "noll_index": noll_j, "mode_name": mode_name,
                        "magnitude_waves_rms": mag,
                        "psnr": res["psnr"], "ssim": res["ssim"],
                        "nmse": res["nmse"], "mse": res["mse"],
                        "aberrated_psnr": res["aberrated_psnr"], "aberrated_ssim": res["aberrated_ssim"],
                        "aberrated_nmse": res["aberrated_nmse"], "aberrated_mse": res["aberrated_mse"],
                        "naive_aberrated_psnr": res["naive_aberrated_psnr"],
                        "naive_aberrated_ssim": res["naive_aberrated_ssim"],
                        "naive_aberrated_nmse": res["naive_aberrated_nmse"],
                        "naive_aberrated_mse": res["naive_aberrated_mse"],
                        "naive_clean_psnr": naive_clean_metrics["psnr"],
                        "naive_clean_nmse": naive_clean_metrics["nmse"],
                        "best_iteration": res["best_iteration"],
                        "seconds": dt,
                    }
                    all_results.append(rec)
                    print(f"  [per-mode] {mode_name:16s} {mag:.2f}w -> "
                          f"CITL(clean)={res['psnr']:.2f}dB  "
                          f"CITL(true)={res['aberrated_psnr']:.2f}dB  "
                          f"naive(true)={res['naive_aberrated_psnr']:.2f}dB  "
                          f"({dt:.1f}s)")

                    if want_visual:
                        fig_dir = os.path.join(experiment.dir, "figures", stem)
                        os.makedirs(fig_dir, exist_ok=True)
                        fig_path = os.path.join(
                            fig_dir, f"per_mode_{mode_name}_{mag:.2f}w.png"
                        )
                        save_condition_figure(
                            target_eval_np, res["naive_aberrated_view_np"],
                            res["aberrated_view_np"],
                            mag, res["aberrated_psnr"], res["aberrated_nmse"],
                            res["naive_aberrated_psnr"], res["naive_aberrated_nmse"],
                            fig_path, mode_name,
                        )

        # ---- Combined sweep ----
        if COMBINED_ENABLED:
            for mag in MAGNITUDES:
                want_visual = (
                    SAVE_VISUALS and VISUAL_SAVE_COMBINED
                    and stem in VISUAL_IMAGE_STEMS and mag in VISUAL_MAGNITUDES
                )
                aberr = aberration_phase_map(
                    grid_size=H, magnitude_waves_rms=mag,
                    noll_indices=tuple(COMBINED_NOLL), combined_equal_rms=True,
                )
                t0 = time.perf_counter()
                res = run_one_condition(
                    target_optim_np, target_eval_np, aberr,
                    naive_phase_np=naive_phase_np,
                )
                dt = time.perf_counter() - t0
                rec = {
                    "image": stem, "sweep": "combined",
                    "noll_index": None,
                    "mode_name": f"combined_{'_'.join(str(j) for j in COMBINED_NOLL)}",
                    "magnitude_waves_rms": mag,
                    "psnr": res["psnr"], "ssim": res["ssim"],
                    "nmse": res["nmse"], "mse": res["mse"],
                    "aberrated_psnr": res["aberrated_psnr"], "aberrated_ssim": res["aberrated_ssim"],
                    "aberrated_nmse": res["aberrated_nmse"], "aberrated_mse": res["aberrated_mse"],
                    "naive_aberrated_psnr": res["naive_aberrated_psnr"],
                    "naive_aberrated_ssim": res["naive_aberrated_ssim"],
                    "naive_aberrated_nmse": res["naive_aberrated_nmse"],
                    "naive_aberrated_mse": res["naive_aberrated_mse"],
                    "naive_clean_psnr": naive_clean_metrics["psnr"],
                    "naive_clean_nmse": naive_clean_metrics["nmse"],
                    "best_iteration": res["best_iteration"],
                    "seconds": dt,
                }
                all_results.append(rec)
                print(f"  [combined] total {mag:.2f}w -> "
                      f"CITL(clean)={res['psnr']:.2f}dB  "
                      f"CITL(true)={res['aberrated_psnr']:.2f}dB  "
                      f"naive(true)={res['naive_aberrated_psnr']:.2f}dB  "
                      f"({dt:.1f}s)")

                if want_visual:
                    fig_dir = os.path.join(experiment.dir, "figures", stem)
                    os.makedirs(fig_dir, exist_ok=True)
                    fig_path = os.path.join(fig_dir, f"combined_{mag:.2f}w.png")
                    save_condition_figure(
                        target_eval_np, res["naive_aberrated_view_np"],
                        res["aberrated_view_np"],
                        mag, res["aberrated_psnr"], res["aberrated_nmse"],
                        res["naive_aberrated_psnr"], res["naive_aberrated_nmse"],
                        fig_path, "combined",
                    )

        # incremental save after each image (so a long run is crash-resilient)
        with open(os.path.join(experiment.dir, "zernike_stress_results.json"), "w") as f:
            json.dump(all_results, f, indent=2)

    print(f"\nDone. {len(all_results)} conditions. Results in {experiment.dir}")
    return all_results


if __name__ == "__main__":
    main()


---

# EXPERIMENTAL

---

In [ ]:
"""
camera_feedback_gd.py

Per-iteration camera-in-the-loop gradient descent across ALL targets listed in
exposure_settings.json - full iteration sweep, every iteration captured, one
target at a time.

Gives gradient descent direct access to the camera's measured intensity at every
iteration, substituted into its own existing, unmodified algorithm (see gd.py's
docstring for the camera_source hook this relies on). No learned model, no phase
corrector, no optimizer over weights.

Alignment calibration
----------------------
Camera<->target alignment is found ONCE per RIG (not per target), automatically
or manually depending on config.hardware.calibration_mode:
  - "auto": quick SIMULATED GD burn-in (no camera) on the first target in
    exposure_settings.json to get a phase pattern whose reconstruction
    resembles something real, one capture, fit the transform (auto_align.py).
  - "manual": build the transform directly from config.hardware.manual_alignment
    (x0/y0/w/h/rotation_deg) - no burn-in, no capture, no fitting. Use this
    once you've physically flattened the rig by hand.
Either way, the resulting transform is FROZEN and reused for every target's
sweep in this run - camera and optics are physically fixed for the whole
session, only exposure (ISO/Tv) changes per target.

Why the transform is frozen, not refit every iteration/target: it represents
a fixed physical property of the rig, not something that should track the
optimization or vary by target content. Refitting per-iteration would let an
8-DOF homography partially absorb loss reduction by warping the comparison
coordinates instead of the optimizer genuinely improving the phase, silently
invalidating every downstream metric. If you suspect the rig has physically
drifted mid-run, re-run calibration as a fresh, separate step and compare.

Per-target exposure
--------------------
exposure_settings.json (see that file's own _readme) is keyed by target
stem, with default_iso resolved via null-inherit and a per-target tv. Values
are human-readable; exposure_lookup.py converts them to the EDSDK hex codes
canon_camera.py's set_iso/set_tv expect. Since this rig has no lens, there is
no Av control - ISO and Tv are the only two exposure knobs set here.
"""

from datetime import datetime
import os
import time
import json
import shutil
from pathlib import Path

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import glob

from canon_camera import CanonEDSDKCamera
from holoeye import slmdisplaysdk

from config_handler import ConfigHandler
from experiment_manager import ExperimentManager
from hardware import tile_to_slm_centered
from stats_torch import normalize_intensity_sum, phase_to_uint8
from physics import hologram_intensity_from_phase
from gd import GradientDescentHologram

from exposure_lookup import iso_to_hex, tv_to_hex


from citl_capture import (
    rename_capture, circular_mask_np, find_dc_center_np,
    normalize_intensity_sum_excluding_mask, load_camera_capture_for_citl,
)


def find_target_image(image_files_dir, stem, exts=(".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")):
    """Resolve a target's actual file path from its stem (exposure_settings.json
    key) by searching image_files_dir for stem + each candidate extension.
    Case-sensitive, exact stem match only - fixes a mismatch here rather than
    guessing, since a silent fuzzy match could load the wrong target."""
    for ext in exts:
        candidate = os.path.join(image_files_dir, stem + ext)
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        f"No image file found for stem {stem!r} in {image_files_dir} "
        f"(tried extensions: {exts}). Check exposure_settings.json's target "
        f"keys match the actual filenames exactly (case-sensitive, no extension)."
    )

# -----------------------------
# Simulation loading helpers
# -----------------------------
def load_warm_start_phase(sim_base_dir, stem, pad_factor, method):
    """
    Loads the iteration-750 SLM-plane phase (H,W) from the matching
    simulation_comparison run for this target/method, at the SAME
    pad_factor as this CITL run - matched-pair by construction, since
    pad_factor is always B2's own PAD_FACTOR, never independently set.
    """
 
    if pad_factor > 1:
        suffix = "faithful"
    else:
        suffix = "ideal"
 
    pad_dir = os.path.join(sim_base_dir, f"simulation_comparison_{suffix}")
    matches = glob.glob(os.path.join(pad_dir, f"{stem}_simulation_comparison*"))
    if not matches:
        raise FileNotFoundError(
            f"No simulation_comparison run found for target {stem!r} under {pad_dir}."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple simulation_comparison runs found for {stem!r} under {pad_dir}: "
            f"{matches}. Remove stale reruns before continuing."
        )
    phase_path = os.path.join(matches[0], "outputs", method, f"{method}_iter_750_phase.npy")
    if not os.path.exists(phase_path):
        raise FileNotFoundError(f"Expected warm-start phase at {phase_path} but it does not exist.")
    return np.load(phase_path).astype(np.float32)


# -----------------------------
# Config + experiment logging
# -----------------------------
config_dir = str(CONFIG_DIR)
print("Config path:", config_dir)

config_file = "camera_feedback_gd"
config = ConfigHandler.load(config_file, search_paths=[config_dir])

save_base = paths.resolve(config.paths.save_base)
save_base.mkdir(parents=True, exist_ok=True)
image_files_dir = paths.resolve(config.paths.image_files)
exposure_settings_path = paths.resolve(config.paths.exposure_settings)
sim_comparison_base = paths.resolve(config.paths.simulation_comparison_base)

with open(exposure_settings_path) as f:
    exposure = json.load(f)

target_stems = list(exposure["targets"].keys())
# target_stems = ["guy", "tram"]
print(f"Loaded exposure settings for {len(target_stems)} targets: {target_stems}")

source_config_path = os.path.join(config_dir, config_file + ".json")

# -----------------------------
# Hardware + run settings (rig-wide, shared across every target)
# -----------------------------
SLM_SHAPE = tuple(config.hardware.slm_shape)
DC_RADIUS = config.hardware.dc_radius
DC_AUTO_CENTER = config.hardware.dc_auto_center
DC_CENTER = tuple(config.hardware.dc_center)
SETTLE_TIME_S = config.hardware.settle_time_s
CAPTURE_TIMEOUT_S = config.hardware.capture_timeout_s
DLL_PATH = config.hardware.dll_path

H = config.run.height
W = config.run.width
GD_ITERATIONS = config.run.gd_iterations
TARGET_FORMULATION = config.run.target_formulation

CALIB_BURNIN_ITERATIONS = getattr(config.run, "calib_burnin_iterations", 200)

SAVE_ALL_ITERATION_PNGS = getattr(config.run, "save_all_iteration_pngs", False)

# -----------------------------
# Flatten conditions x init_variants into one ordered list of run configs --
# simpler to loop over than nested condition/init_variant loops, and this
# list doubles as the bookkeeping record of what actually ran (see summary
# printed at the very end).
# -----------------------------
run_configs = []
for condition in config.conditions:
    for init_variant in condition.init_variants:
        run_configs.append({
            "physics_name": condition.physics_name,
            "pad_factor": condition.pad_factor,
            "apply_sinc": condition.apply_sinc,
            "fill_factor": condition.fill_factor,
            "fill_is_areal": condition.fill_is_areal,
            "init_name": init_variant.init_name,
            "gd_init_phase": init_variant.gd_init_phase,
        })
print(f"Loaded {len(run_configs)} run configs: " +
      ", ".join(f"{rc['init_name']}/{rc['physics_name']}" for rc in run_configs))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

np.random.seed(0)
torch.manual_seed(0)


def phase_to_slm_frame(phase_np_or_t):
    if isinstance(phase_np_or_t, torch.Tensor):
        phase_np = phase_np_or_t.detach().cpu().numpy().astype(np.float32)
    else:
        phase_np = np.asarray(phase_np_or_t, dtype=np.float32)
    phase_8bit = phase_to_uint8(phase_np)
    holo_small = np.fliplr(phase_8bit).copy()
    holo_slm = tile_to_slm_centered(holo_small, slm_shape=SLM_SHAPE)
    return np.ascontiguousarray(holo_slm.astype(np.uint8))


def load_and_prepare_target(image_path):
    img = Image.open(image_path).convert("L")
    X = np.asarray(img, dtype=np.float32)
    if X.shape != (H, W):
        raise ValueError(f"Target image shape {X.shape} != expected ({H},{W}) for {image_path}")

    X_clipped = np.clip(X, 0.0, None)
    if TARGET_FORMULATION == "amplitude":
        target_optim_np = np.sqrt(X_clipped).astype(np.float32)
    else:
        target_optim_np = X_clipped.astype(np.float32)

    return X, target_optim_np


try:
    slm = slmdisplaysdk.SLMInstance()
    if not slm.requiresVersion(5):
        raise RuntimeError("Required SDK version not available.")
    error = slm.open()
    if error != slmdisplaysdk.ErrorCode.NoError:
        raise RuntimeError(slm.errorString(error))
    print("SLM opened.")

    camera = CanonEDSDKCamera(
        dll_path=DLL_PATH,
        save_dir=os.path.join(save_base, "_rig_calibration", "dslr captures"),
        auto_set_save_to_host=True,
        auto_set_capacity=True,
        verbose=True,
    )
    camera.initialize()
    camera.open_session()
    print("Camera opened.")

    camera.assert_manual_mode()
    print("Confirmed: camera is in Manual (M) exposure mode.")

    rig_calib_dir = os.path.join(save_base, "_rig_calibration")
    os.makedirs(rig_calib_dir, exist_ok=True)

    CALIBRATION_MODE = config.hardware.calibration_mode  # "auto" or "manual"
    print(f"\n=== Alignment calibration (mode={CALIBRATION_MODE}) ===")
    t_calib0 = time.perf_counter()

    if CALIBRATION_MODE == "manual":
        m = config.hardware.manual_alignment
        CAMERA_ROI = {"y0": m.y0, "x0": m.x0, "h": m.h, "w": m.w}
        ROTATE_DEGREES = m.rotation_deg

        calib_json_path = os.path.join(rig_calib_dir, "calibration.json")
        with open(calib_json_path, "w") as f:
            json.dump({
                "mode": "manual",
                "roi": CAMERA_ROI,
                "rotation_deg": ROTATE_DEGREES,
            }, f, indent=2)
        print(
            f"Manual alignment: x0={m.x0}, y0={m.y0}, w={m.w}, h={m.h}, "
            f"rotation_deg={m.rotation_deg}. Saved to: {calib_json_path}"
        )
        calib = None

    else:
        # Auto mode fitted a homography via auto_align.calibrate(). It has no
        # equivalent under the roi+angle (similarity-transform) calibration this
        # file now uses. Fail loudly rather than silently mismatch.
        raise NotImplementedError(
            "CALIBRATION_MODE='auto' is not supported under the old "
            "roi+angle calibration method (citl_capture.py) - it fit a "
            "homography, which the similarity-transform approach cannot "
            "represent in general. Use CALIBRATION_MODE='manual'."
        )

    t_calib1 = time.perf_counter()
    print(f"Calibration done in {t_calib1 - t_calib0:.1f}s. Transform frozen for the rest of this run "
          f"(rig-wide, independent of physics condition under the old roi+angle calibration method).")

    # ========================================================================
    # OUTER LOOP - one iteration per (physics condition, init variant)
    # combination, i.e. one of the 4 rows in run_configs. Calibration above
    # is genuinely shared across all 4; everything physics-dependent
    # (pad_factor-derived quantities, dc_mask_static) is rebuilt fresh here.
    # ========================================================================
    run_summary = []

    for run_config in run_configs:
        PAD_FACTOR = run_config["pad_factor"]
        APPLY_SINC = run_config["apply_sinc"]
        FILL_FACTOR = run_config["fill_factor"]
        FILL_IS_AREAL = run_config["fill_is_areal"]
        GD_INIT_PHASE_SOURCE = run_config["gd_init_phase"]

        M_PAD, N_PAD = H * PAD_FACTOR, W * PAD_FACTOR
        OUT_HW_LOSS = (M_PAD, N_PAD)
        DC_CENTER_PADDED = (DC_CENTER[0] * PAD_FACTOR, DC_CENTER[1] * PAD_FACTOR)
        DC_RADIUS_PADDED = DC_RADIUS * PAD_FACTOR

        dc_mask_static = torch.from_numpy(
            circular_mask_np(OUT_HW_LOSS, DC_CENTER_PADDED[0], DC_CENTER_PADDED[1], DC_RADIUS_PADDED)
        ).to(device=device, dtype=torch.bool)

        save_path = os.path.join(save_base, run_config["init_name"], f"citl_gs_gd_{run_config['physics_name']}")
        os.makedirs(save_path, exist_ok=True)

        print(
            f"\n{'=' * 70}\n"
            f"# RUN CONFIG: {run_config['init_name']} / {run_config['physics_name']} "
            f"(pad_factor={PAD_FACTOR}) -- {len(target_stems)} targets\n"
            f"{'=' * 70}"
        )
        t_run_start = time.perf_counter()

        for target_idx, stem in enumerate(target_stems):
            print(f"\n{'#' * 70}\n# TARGET {target_idx + 1}/{len(target_stems)}: {stem}\n{'#' * 70}")

            image_path = find_target_image(image_files_dir, stem)

            experiment = ExperimentManager(
                name=f"{stem}_{run_config['physics_name']}",
                base_dir=save_path,
                overwrite=config.experiment.overwrite,
            )

            camera_dir = os.path.join(experiment.dir, "dslr captures")
            out_dir = os.path.join(experiment.dir, "outputs")
            processed_dir = os.path.join(experiment.dir, "processed")
            gd_captures_dir = os.path.join(processed_dir, "gd_captures")
            for d in (camera_dir, out_dir, processed_dir, gd_captures_dir):
                os.makedirs(d, exist_ok=True)

            camera.save_dir = Path(camera_dir)

            timestamp = datetime.now()
            experiment.log(f"Experiment Details:\r\nExperiment: {config.experiment.description}")
            experiment.log(f"Date: {timestamp}")
            experiment.log(f"Image: {image_path}")
            experiment.log(f"GD Iterations: {GD_ITERATIONS}")
            experiment.log(f"Target formulation: {TARGET_FORMULATION}")
            experiment.log(
                f"Forward model: pad_factor={PAD_FACTOR}, apply_sinc={APPLY_SINC}, "
                f"fill_factor={FILL_FACTOR}, fill_is_areal={FILL_IS_AREAL} "
                f"-> loss/capture resolution ({M_PAD},{N_PAD})"
            )
            if CALIBRATION_MODE == "manual":
                experiment.log(f"Calibration: manual (rig-wide, see {rig_calib_dir}/calibration.json)")
            else:
                experiment.log(f"Calibration: auto (rig-wide, see {rig_calib_dir}/calibration.json)")

            dest_config_path = os.path.join(experiment.dir, "run_configuration.json")
            shutil.copy2(source_config_path, dest_config_path)
            experiment.log(f"Saved config snapshot to: {dest_config_path}")

            exp_entry = exposure["targets"][stem]
            iso_val = exp_entry["iso"] if exp_entry["iso"] is not None else exposure["default_iso"]
            tv_val = exp_entry["tv"]
            iso_hex = iso_to_hex(iso_val)
            tv_hex = tv_to_hex(tv_val)
            camera.set_iso(iso_hex)
            camera.set_tv(tv_hex)
            experiment.log(f"Exposure: ISO={iso_val} (0x{iso_hex:08X}), Tv={tv_val} (0x{tv_hex:02X})")

            X, target_optim_np = load_and_prepare_target(image_path)
            X_batched = X.reshape(1, H, W)
            target_eval_t = torch.from_numpy(X_batched).to(device=device, dtype=torch.float32)
            target_eval_t = normalize_intensity_sum(target_eval_t)
            target_eval_np = target_eval_t[0].detach().cpu().numpy().astype(np.float32)

            plt.figure()
            plt.imshow(X, cmap="gray")
            plt.title(f"Target: {stem}")
            plt.axis("off")
            plt.show()

            gd_seed = (load_warm_start_phase(sim_comparison_base, stem, PAD_FACTOR, "gd")
                       if GD_INIT_PHASE_SOURCE == "warm_start" else GD_INIT_PHASE_SOURCE)

            iteration_counter = {"gd": 0}
            t_start = time.perf_counter()

            timing = {
                "gd": {"slm": [], "capture": [], "load": [], "total": []},
            }

            def capture_for_phase(phase_np_or_t, method_label, iteration_index, eps_norm=1e-12):
                t_iter0 = time.perf_counter()
                holo_slm = phase_to_slm_frame(phase_np_or_t)

                t_slm0 = time.perf_counter()
                error = slm.showData(holo_slm)
                if error != slmdisplaysdk.ErrorCode.NoError:
                    raise RuntimeError(f"SLM error: {slm.errorString(error)}")
                time.sleep(SETTLE_TIME_S)
                t_slm1 = time.perf_counter()

                t_cam0 = time.perf_counter()
                result = camera.capture_image(timeout_s=CAPTURE_TIMEOUT_S)
                t_cam1 = time.perf_counter()

                renamed_path = rename_capture(
                    result.path,
                    capture_dir=camera_dir,
                    stem=f"{method_label}_iter_{iteration_index:04d}",
                )
                gd_raw_paths[iteration_index] = renamed_path

                t_load0 = time.perf_counter()

                I_cam_t, cam_resized_np, dc_mask_t, dc_center_used = load_camera_capture_for_citl(
                    image_path=renamed_path,
                    roi=CAMERA_ROI,
                    angle=ROTATE_DEGREES,
                    out_hw=OUT_HW_LOSS,
                    device=device,
                    eps_norm=eps_norm,
                    dc_radius=DC_RADIUS_PADDED,
                    auto_center=DC_AUTO_CENTER,
                    dc_center=DC_CENTER_PADDED,
                    subtract_min=True,
                    median_ksize=0,
                )
                processed_path = os.path.join(gd_captures_dir, f"{method_label}_iter_{iteration_index:04d}.png")
                Image.fromarray(
                    np.clip(cam_resized_np / (cam_resized_np.max() + 1e-8) * 255, 0, 255).astype(np.uint8)
                ).save(processed_path)
                gd_processed_paths[iteration_index] = processed_path
                t_load1 = time.perf_counter()

                total_iters = GD_ITERATIONS
                if iteration_index % 50 == 0 or iteration_index == total_iters - 1:
                    experiment.log(f"[{method_label}] iter {iteration_index}: captured frame, dc_center={dc_center_used}")

                t_iter1 = time.perf_counter()
                timing[method_label]["slm"].append(t_slm1 - t_slm0)
                timing[method_label]["capture"].append(t_cam1 - t_cam0)
                timing[method_label]["load"].append(t_load1 - t_load0)
                timing[method_label]["total"].append(t_iter1 - t_iter0)

                return I_cam_t

            print(f"\n=== GD (camera feedback) [{stem}] ===")
            experiment.log("=== GD (camera feedback) start ===")
            t_gd0 = time.perf_counter()

            def gd_camera_source(phase_for_forward_t, iteration_index):
                I_cam_t = capture_for_phase(phase_for_forward_t, "gd", iteration_index)
                iteration_counter["gd"] = iteration_index
                return I_cam_t

            gd_raw_paths = {}
            gd_best_state = {"path": None, "iteration": None}
            gd_prune_stats = {"deleted": 0, "kept": 0}

            gd_processed_paths = {}
            gd_processed_best_state = {"path": None, "iteration": None}
            gd_processed_prune_stats = {"deleted": 0, "kept": 0}

            def gd_iteration_callback(record):
                k = record["iteration"]
                is_first = (k == 0)
                is_last = (k == GD_ITERATIONS - 1)

                raw_path = gd_raw_paths.get(k)
                if raw_path is not None:
                    if record["is_best"]:
                        old_path, old_iter = gd_best_state["path"], gd_best_state["iteration"]
                        if old_path is not None and old_iter != 0 and old_path != raw_path and os.path.exists(old_path):
                            os.remove(old_path)
                            gd_prune_stats["deleted"] += 1
                        gd_best_state["path"], gd_best_state["iteration"] = raw_path, k
                        gd_prune_stats["kept"] += 1
                    elif is_first or is_last:
                        gd_prune_stats["kept"] += 1
                    else:
                        if os.path.exists(raw_path):
                            os.remove(raw_path)
                            gd_prune_stats["deleted"] += 1

                if not SAVE_ALL_ITERATION_PNGS:
                    processed_path = gd_processed_paths.get(k)
                    if processed_path is not None:
                        if record["is_best"]:
                            old_path, old_iter = gd_processed_best_state["path"], gd_processed_best_state["iteration"]
                            if old_path is not None and old_iter != 0 and old_path != processed_path and os.path.exists(old_path):
                                os.remove(old_path)
                                gd_processed_prune_stats["deleted"] += 1
                            gd_processed_best_state["path"], gd_processed_best_state["iteration"] = processed_path, k
                            gd_processed_prune_stats["kept"] += 1
                        elif is_first or is_last:
                            gd_processed_prune_stats["kept"] += 1
                        else:
                            if os.path.exists(processed_path):
                                os.remove(processed_path)
                                gd_processed_prune_stats["deleted"] += 1

            gd_log_csv = os.path.join(experiment.dir, "gd_camera_feedback_convergence.csv")

            gd_out = GradientDescentHologram().run(
                target_optim_np,
                iterations=GD_ITERATIONS,
                init_phase=gd_seed,
                lr=config.run.gd_lr,
                optimizer_name=config.run.gd_optimizer_name,
                tv_weight=config.run.gd_tv_weight,
                device=device,
                target_formulation=TARGET_FORMULATION,
                camera_source=gd_camera_source,
                dc_mask=dc_mask_static,
                log_csv_path=gd_log_csv,
                pad_factor=PAD_FACTOR,
                apply_sinc=APPLY_SINC,
                fill_factor=FILL_FACTOR,
                fill_is_areal=FILL_IS_AREAL,
                iteration_callback=gd_iteration_callback,
            )

            experiment.log(
                f"Raw capture pruning (GD): kept {gd_prune_stats['kept']}, deleted {gd_prune_stats['deleted']} "
                f"(target: first + last + running-best only, i.e. 3 raw files retained)"
            )
            if SAVE_ALL_ITERATION_PNGS:
                experiment.log("Processed PNG pruning (GD): disabled, kept every iteration (SAVE_ALL_ITERATION_PNGS=True)")
            else:
                experiment.log(
                    f"Processed PNG pruning (GD): kept {gd_processed_prune_stats['kept']}, "
                    f"deleted {gd_processed_prune_stats['deleted']} (first + last + running-best only)"
                )

            t_gd1 = time.perf_counter()
            np.save(os.path.join(out_dir, "gd_camera_feedback_phase.npy"), gd_out["phase_radians"])
            Image.fromarray(gd_out["phase_uint8"]).save(os.path.join(out_dir, "gd_camera_feedback_phase_8bit.png"))

            experiment.log(f"GD convergence CSV saved to: {gd_log_csv}")

            best_iteration = gd_out["best_iteration"]
            if best_iteration is None:
                raise RuntimeError(
                    f"[{stem}] GD's best_iteration is None (loss never improved from its "
                    f"initial value) - cannot identify which camera capture corresponds "
                    f"to the returned phase. Check gd.run()'s loss computation before "
                    f"trusting this target's result."
                )

            best_capture_path = os.path.join(gd_captures_dir, f"gd_iter_{best_iteration:04d}.png")
            if not os.path.exists(best_capture_path):
                raise FileNotFoundError(
                    f"[{stem}] Expected saved capture for GD's best iteration ({best_iteration}) "
                    f"at {best_capture_path}, but it does not exist. Capture saving and "
                    f"best-iteration tracking may be out of sync - check capture_for_phase "
                    f"and gd.py's best_iteration bookkeeping before trusting this result."
                )

            experiment.log(
                f"GD best iteration: {best_iteration} "
                f"(internal training loss at this iteration = {gd_out['best_loss']:.6e}, "
                f"on gd.py's internal sum-normalized*H*W, unweighted, dc-excluded scale). "
                f"Best capture saved at: {best_capture_path}"
            )
            experiment.log(f"=== GD (camera feedback) done in {t_gd1 - t_gd0:.1f}s ===")
            print(f"GD (camera feedback) done for {stem}. Best capture:", best_capture_path)

            plt.figure(figsize=(4, 4))
            plt.imshow(Image.open(best_capture_path).convert("L"), cmap="gray")
            plt.title(f"GD (camera feedback) best capture (iter {best_iteration}) [{stem}]")
            plt.axis("off")
            plt.show()

            def summarize_timing(label, values):
                if not values:
                    return f"{label}: no data"
                arr = np.array(values)
                return (f"{label}: mean={arr.mean():.3f}s | std={arr.std():.3f}s | "
                        f"min={arr.min():.3f}s | max={arr.max():.3f}s | n={len(arr)}")

            experiment.log("\n=== Timing summary ===")
            for method_label in ("gd",):
                for category in ("slm", "capture", "load", "total"):
                    experiment.log(f"[{method_label}] " + summarize_timing(category, timing[method_label][category]))

            gd_total_hr = (t_gd1 - t_gd0) / 3600
            experiment.log(
                f"\nHeadline: GD camera-feedback sweep took "
                f"{gd_total_hr:.2f} hr ({GD_ITERATIONS} iterations, {stem})."
            )

            run_elapsed_s = time.perf_counter() - t_start
            msg = "=" * 60 + "\n"
            msg += f"Experiment: {config.experiment.description}\n"
            msg += f"Target: {stem}\n"
            msg += f"Date: {timestamp}\n"
            msg += f"Image: {image_path}\n"
            msg += f"Exposure: ISO={iso_val} (0x{iso_hex:08X}), Tv={tv_val} (0x{tv_hex:02X})\n"
            if CALIBRATION_MODE == "manual":
                msg += f"Calibration: manual (rig-wide, see {rig_calib_dir}/calibration.json)\n"
            else:
                msg += (f"Calibration: auto (rig-wide, reference={calib_target_stem}, "
                         f"see {rig_calib_dir}/calibration.json)\n")
            msg += f"GD Iterations: {GD_ITERATIONS}, Target formulation: {TARGET_FORMULATION}\n"
            msg += f"Total runtime: {run_elapsed_s/60:.1f} min ({run_elapsed_s/3600:.2f} hr)\n"
            msg += f"GD camera-feedback wall-clock: {gd_total_hr:.2f} hr for {GD_ITERATIONS} iterations\n"
            msg += f"GD: best iteration {best_iteration} (training loss {gd_out['best_loss']:.6e}), capture at {best_capture_path}\n"
            msg += "No metrics computed here -- scoring is done by the analysis notebook.\n"
            msg += "=" * 60 + "\n\n"

            experiment.write_summary(msg)
            experiment.prepend("Summary\r\n" + msg)
            print(msg)

        t_run_elapsed_hr = (time.perf_counter() - t_run_start) / 3600
        print(f"\nRun config {run_config['init_name']}/{run_config['physics_name']} complete: "
              f"{len(target_stems)} targets in {t_run_elapsed_hr:.2f} hr.")
        run_summary.append({
            "init_name": run_config["init_name"],
            "physics_name": run_config["physics_name"],
            "targets_completed": len(target_stems),
            "elapsed_hr": t_run_elapsed_hr,
        })

    # ========================================================================
    # BOOKKEEPING - which of the run_configs actually completed, and how
    # long each took. Printed once at the very end, after all conditions x
    # init_variants x targets are done.
    # ========================================================================
    print(f"\n{'=' * 70}\nALL RUN CONFIGS COMPLETE ({len(run_summary)}/{len(run_configs)})\n{'=' * 70}")
    for rs in run_summary:
        print(f"  {rs['init_name']:12s} / {rs['physics_name']:10s} -- "
              f"{rs['targets_completed']} targets, {rs['elapsed_hr']:.2f} hr")
    total_hr = sum(rs["elapsed_hr"] for rs in run_summary)
    print(f"Total wall-clock across all run configs: {total_hr:.2f} hr")

finally:
    try:
        slm.close()
    except Exception as e:
        print("SLM close error (may already be closed):", e)
    try:
        camera.close()
    except Exception as e:
        print("Camera close error (may already be closed):", e)
    print("Closed camera and SLM.")